In [0]:
%run "/Workspace/Users/pateldharmilkumar@gmail.com/banking-realtime-project/Batchdata/databricks/bronze/Connection"

In [0]:
from pyspark.sql.functions import *

In [0]:
df_customers = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")

    .option("cloudFiles.useNotifications", "true")
    .option("cloudFiles.subscriptionId", "4967f902-d637-449d-81c4-901cd225af2f")
    .option("cloudFiles.tenantId", tenant_id)
    .option("cloudFiles.clientId", client_id)
    .option("cloudFiles.clientSecret", secret_value)
    .option("cloudFiles.resourceGroup", "Realtimedata")
    .option("cloudFiles.queueName", "autoloader-customers-queue")

    .option("cloudFiles.schemaLocation", "/Volumes/bankingdata/bronze/schemalocation/customers/")
    .option("badRecordsPath", "/Volumes/bankingdata/bronze/badrecords/customers/")
    .option("pathGlobFilter", "*.csv")

    .load("abfss://project2@project1demo.dfs.core.windows.net/customers/")
)

df_customers = df_customers.withColumn("ingestion_time", current_timestamp()) \
                           .withColumn("source_file", input_file_name())

df_customers.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/bankingdata/bronze/checkpoints/customers/") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("bankingdata.bronze.customers")